[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/identity-anonymizer/blob/master/notebooks/03_inference_image.ipynb)

Colab上で開く場合，このnotebookはColab用の環境構築セルを含まないため，先に [demo.ipynb](../demo.ipynb) の「1. Python環境のセットアップ」「2. リポジトリの取得と依存関係のインストール」と同様の手順を実行しておくこと．環境構築を含めて手軽に匿名化を試すだけであれば，[demo.ipynb](../demo.ipynb) の利用を推奨する．

# 03: 画像1枚に対する匿名化のデモ

`FaceAnonymizerPipeline` を用いて，画像1枚に写る人物の顔を，属性(性別・年齢等)や表情を保持した
まま架空の人物へ匿名化する．スケール係数(`noise_level`)を変えることで，匿名性(元の顔との
類似度の低さ)と属性の一貫性のトレードオフを確認する．


In [ ]:
import os
import sys

# notebooks/ から見て1階層上がリポジトリルート
REPO_ROOT = os.path.dirname(os.getcwd())
sys.path.insert(0, os.path.join(REPO_ROOT, "src"))
os.chdir(REPO_ROOT)

import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_jp_font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
if os.path.exists(_jp_font_path):
    fm.fontManager.addfont(_jp_font_path)
    plt.rcParams["font.family"] = "Noto Sans CJK JP"
    plt.rcParams["axes.unicode_minus"] = False


In [ ]:
from identity_anonymizer.faceswap import load_ghost_models, FaceAnonymizerPipeline
from identity_anonymizer.anonymizers import get_anonymizer

VAE_WEIGHT_PATH = "weights/anonymizers/vae/vae_512_128.pt"

models = load_ghost_models()
anonymizer = get_anonymizer("vae", weight_path=VAE_WEIGHT_PATH)
pipeline = FaceAnonymizerPipeline(models, anonymizer)


In [ ]:
TARGET_IMAGE_PATH = "sample_images/beckham.jpg"  # 任意の画像パスに変更できる
NOISE_LEVELS = [0.0, 1.0, 1.25, 1.5, 2.0]

import cv2
from utils.inference.image_processing import crop_face

original_full = cv2.imread(TARGET_IMAGE_PATH)
original_face = crop_face(original_full, pipeline.models.app, pipeline.crop_size)[0]

fig, axes = plt.subplots(1, len(NOISE_LEVELS) + 1, figsize=(3 * (len(NOISE_LEVELS) + 1), 3))

axes[0].imshow(original_face[:, :, ::-1])
axes[0].set_title("Original")
axes[0].axis("off")

for ax, noise_level in zip(axes[1:], NOISE_LEVELS):
    face_image, _ = pipeline.anonymize_image(TARGET_IMAGE_PATH, noise_level=noise_level)
    ax.imshow(face_image[:, :, ::-1])
    ax.set_title(f"noise_level={noise_level}")
    ax.axis("off")

fig.tight_layout()
os.makedirs("outputs", exist_ok=True)
fig.savefig("outputs/noise_level_comparison.png")
plt.show()


## 複数画像への一括適用

`sample_images/` 内の全画像に対して，`noise_level=1.25` で匿名化した結果をまとめて確認する．


In [ ]:
import os as _os

sample_image_paths = [
    _os.path.join("sample_images", name)
    for name in sorted(_os.listdir("sample_images"))
    if name.lower().endswith((".jpg", ".jpeg", ".png"))
]

fig, axes = plt.subplots(2, len(sample_image_paths), figsize=(3 * len(sample_image_paths), 6))
for col, image_path in enumerate(sample_image_paths):
    original_full = cv2.imread(image_path)
    original_face = crop_face(original_full, pipeline.models.app, pipeline.crop_size)[0]
    face_image, _ = pipeline.anonymize_image(image_path, noise_level=1.25)

    axes[0, col].imshow(original_face[:, :, ::-1])
    axes[0, col].set_title(_os.path.basename(image_path))
    axes[0, col].axis("off")

    axes[1, col].imshow(face_image[:, :, ::-1])
    axes[1, col].axis("off")

fig.tight_layout()
fig.savefig("outputs/batch_comparison.png")
plt.show()
